In [1]:
from agents import Agent, Production, Chat, Toolkit, Prompt
from pydantic import BaseModel

In [2]:
from agents.utils import get_connection

conn = get_connection()
cur = conn.cursor()

for table in ["Usage", "Chat", "Agent"]:
    cur.execute("""
        SELECT TABLE_NAME
        FROM INFORMATION_SCHEMA.Tables
        WHERE TABLE_TYPE='BASE TABLE'
          AND TABLE_SCHEMA='SQLUser'
          AND TABLE_NAME=?
    """, (table,))
    if cur.fetchone():
        cur.execute(f"DROP TABLE IF EXISTS SQLUser.{table}")

conn.commit()

### **Toolkit**

Toolkits are MCP servers that are externally run. Before initializing a Toolkit object, the MCP server needs to be operational.

In [3]:
utils_toolkit = Toolkit(name = 'Utilities', url = 'http://localhost:9001/mcp')
iris_toolkit = Toolkit(name='IRIS', url = 'http://localhost:9002/mcp')


Load started on 04/15/2026 11:35:42
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/15/2026 11:35:42
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/15/2026 11:35:42
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/15/2026 11:35:43
Loading file Agents.Operation.ToolkitUtilities.cls as udl
Compiling class Agents.Operation.ToolkitUtilities
Compiling routine Agents.Operation.ToolkitUtilities.1
Load finished successfully.

Load started on 04/15/2026 11:35:43
Loading file Agents.Message.ToolRequest.cls as ud

### **Chat**

The Chat API can be used to persist conversations. A chat id can be used to construct a history of that Chat from IRIS instead of needing to maintain it manually. This is particularly important when Enterprise licenses for OpenAI have Zero Data Retention enabled and so OpenAI is not authorized to store the conversation on their servers, the Chat API allows for constructing the conversation from history stored in IRIS.

In [4]:
context = Chat(
    name="travel",
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "We are in Washington DC"},
        {"role": "assistant", "content": "Great, what do you want to do in DC?"}
    ]
)
context

Chat(name='travel', messages=3)

In [5]:
context.messages

[{'role': 'system', 'content': 'You are helpful.'},
 {'role': 'user', 'content': 'We are in Washington DC'},
 {'role': 'assistant', 'content': 'Great, what do you want to do in DC?'}]

In [6]:
context == Chat('travel')

True

### **Prompt**

- The Prompt API is a way to manage and version Prompts. 
- Prompts can be built at runtime using parameters. 
- Prompts Prompts versions can be fetched by a selected version. 
- Variables contained in a prompt can be queried using `get_variables()` method.

In [7]:
bond_system = Prompt(name = 'Agent007', text = 'You are {agent_name}. You always stay in character.')
bond_system.build(agent_name='James Bond')

'You are James Bond. You always stay in character.'

In [8]:
bond_system = Prompt(name = 'Agent007', text = 'Your next mission is of utmost importance, you do not have time to talk.')
bond_system

Prompt(name='Agent007', version=2, text='Your next mission is of utmost importance, you do not have time to talk.')

In [9]:
Prompt('Agent007') == bond_system

True

In [10]:
Prompt('Agent007', version=1)

Prompt(name='Agent007', version=1, text='You are {agent_name}. You always stay in character.')

In [11]:
Prompt('Agent007', version=1).get_variables()

['agent_name']

In [12]:
Prompt('Agent007').delete()
try:
    prompt = Prompt("Agent007")
except ValueError as e:
    print(e)

No prompt found for 'Agent007'


### **Agents**

Agents can be defined by a name, a description (not currently used in any way but can be leveraged in the future for expert selection), and an OpenAI model. Optionally, agents can be configured with a default structured output (modifiable at call time) and a set of toolkits the agent should have access to. These tools are advertised to the LLM specific to access the agent has at a Toolkit level (specifying individual tools inside a Toolkit is not currently supported). Agents must be added to a Production before being used.

In [13]:
molly = Agent(name='Molly', model='gpt-5')
Production('AgentSpace', [molly]).start()
molly('What are some summer hiking trails around Boston?')


Load started on 04/15/2026 11:35:45
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/15/2026 11:35:45
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/15/2026 11:35:45
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/15/2026 11:35:45
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/15/2026 11:35:46
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

'Here are great summer hikes near Boston:\n\n- Middlesex Fells Reservation (Medford/Stoneham): Skyline Trail (7–9 mi, rugged, views), shorter loops like Sheepfold and Spot Pond. MBTA: Orange Line to Oak Grove + bus; parking fills on weekends.\n- Blue Hills Reservation (Milton/Quincy): Skyline Trail segments, Great Blue Hill summit tower, Ponkapoag Bog boardwalk. Lots of shade; can be rocky. Red Line to Ashmont + buses; multiple lots.\n- Walden Pond State Reservation (Concord): 1.7-mi pond loop + trails to Thoreau sites. Arrive early; lot fills and sometimes closes mid-day. Commuter Rail to Concord.\n- Minute Man National Historical Park (Lexington/Concord): Battle Road Trail (easy, mostly flat, historic sites). Hot in open sections; bring sun protection.\n- Lynn Woods Reservation (Lynn): Dungeon Rock, Stone Tower, many shaded loops. MBTA buses from Wonderland; free parking.\n- Harold Parker State Forest (Andover/N. Andover): Quiet forest loops around ponds; great for longer, mellow mil

Agents can be fetched using only their name. Adding any other parameters will be treated as agent creation.

In [14]:
Agent('Molly') == molly

True

In [16]:
class AlexResponse(BaseModel):
    message: str
    reasoning: str

class MollyResponse(BaseModel):
    text: str
    reasoning: str

alex = Agent(name='Alex', 
             description='Test Agent 1', 
             system_prompt=Prompt(name='alex_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit],
             response_format=AlexResponse)

molly = Agent(name='Molly', 
             description='Test Agent 2', 
             system_prompt=Prompt(name='molly_system', text='You are a helpful agent'),
             model='gpt-5',
             reasoning_effort='low',
             toolkits=[utils_toolkit, iris_toolkit],
             response_format=MollyResponse)


Load started on 04/15/2026 11:38:11
Loading file Agents.Message.AlexResponse.cls as udl
Compiling class Agents.Message.AlexResponse
Compiling table Agents_Message.AlexResponse
Compiling routine Agents.Message.AlexResponse.1
Load finished successfully.

Load started on 04/15/2026 11:38:11
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/15/2026 11:38:11
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/15/2026 11:38:11
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/15/2026 11:38:12
Loading file Agents.Ope

In [17]:
Production('AgentSpace', [molly, alex]).start()


Load started on 04/15/2026 11:38:21
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/15/2026 11:38:21
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/15/2026 11:38:21
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/15/2026 11:38:21
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/15/2026 11:38:21
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

In [18]:
molly(message='Which tables do we have in IRIS in the Agents namespace?')

'{"text": "The tables in Agents are: SQLUser.Agent, SQLUser.AgentToolkit, SQLUser.Chat, SQLUser.Prompt, SQLUser.TestModel, SQLUser.Toolkit, SQLUser.ToolUsage, SQLUser.Usage.", "reasoning": "Used the existing IRIS list_tables result for the Agents namespace."}'

In [19]:
molly(message='What is the weather today?', chat=context)

'{"text": "Today in Washington, DC: Cloudy, high 26\\u00b0, low 13\\u00b0.", "reasoning": "Used the provided Utilities.weather tool result for Washington, DC and did not recall the tool."}'

In [20]:
molly(message='Recommend some good food spots for lunch', chat='travel', reasoning_effort='high')

'{"text": "Here are great lunch options around Washington, DC:\\n- Old Ebbitt Grill (Downtown): Classic American near the White House; oysters, crab cakes; good for walk-ins if early.\\n- Le Diplomate (Logan Circle): Beloved French brasserie; reservations help.\\n- Zaytinya (Penn Quarter): Mediterranean mezze by Jos\\u00e9 Andr\\u00e9s; great for sharing.\\n- Jaleo (Penn Quarter): Spanish tapas; quick, flavorful small plates.\\n- Founding Farmers (Foggy Bottom): Big American comfort-food menu; good for groups.\\n- Rasika (Penn Quarter/West End): Upscale Indian; famous palak chaat\\u2014check lunch hours.\\n- Shouk (Mt Vernon Triangle/Georgetown): Plant-based Israeli pitas and bowls; fast-casual.\\n- CHIKO (Capitol Hill/Dupont): Creative Chinese-Korean; fast-casual and hearty.\\n- Call Your Mother (Logan Circle/Georgetown): Excellent bagel sandwiches; grab-and-go.\\n- Ben\\u2019s Chili Bowl (U Street): Iconic DC half-smokes; quick and classic.\\n- Teaism (Penn Quarter/Dupont): Light, ta

In [21]:
class Restaurant(BaseModel):
    name: str
    cuisine: str

class TasteAtlas(BaseModel):
    restaurants: list[Restaurant]
    reasoning: str

molly(message='What are some places I would like? I tend to like Italian and Asian cuisines', response_format=TasteAtlas, chat='travel')


Load started on 04/15/2026 12:07:59
Loading file Agents.Message.Restaurant.cls as udl
Compiling class Agents.Message.Restaurant
Compiling routine Agents.Message.Restaurant.1
Load finished successfully.

Load started on 04/15/2026 12:07:59
Loading file Agents.Message.TasteAtlas.cls as udl
Compiling class Agents.Message.TasteAtlas
Compiling table Agents_Message.TasteAtlas
Compiling routine Agents.Message.TasteAtlas.1
Load finished successfully.


'{"restaurants": [{"name": "L\'Ardente", "cuisine": "Italian"}, {"name": "Osteria Morini", "cuisine": "Italian"}, {"name": "Sfoglina", "cuisine": "Italian"}, {"name": "RPM Italian", "cuisine": "Italian"}, {"name": "The Red Hen", "cuisine": "Italian"}, {"name": "Filomena Ristorante", "cuisine": "Italian"}, {"name": "Daikaya", "cuisine": "Japanese"}, {"name": "Sushi Taro", "cuisine": "Japanese"}, {"name": "Rasika", "cuisine": "Indian"}, {"name": "Thip Khao", "cuisine": "Lao"}, {"name": "Anju", "cuisine": "Korean"}, {"name": "Maketto", "cuisine": "Cambodian/Taiwanese"}], "reasoning": "Based on your preference for Italian and Asian cuisines in Washington, DC, here are well-regarded spots across price points and neighborhoods to match those tastes."}'

In [22]:
Chat('travel').usage()

'{"input_tokens": 3043, "output_tokens": 4322, "output_reasoning_tokens": 3456, "total_tokens": 7365}'

In [23]:
Production('AgentSpace').usage()

{'input_tokens': 4723,
 'output_tokens': 6481,
 'output_reasoning_tokens': 4864,
 'total_tokens': 11204}

In [24]:
molly.usage()

{'input_tokens': 5120,
 'output_tokens': 9549,
 'output_reasoning_tokens': 7360,
 'total_tokens': 14669}

In [25]:
Production('AgentSpace').usage(agents=[Agent('Molly')])

{'input_tokens': 4723,
 'output_tokens': 6481,
 'output_reasoning_tokens': 4864,
 'total_tokens': 11204}

In [ ]:
Production('AgentSpace').delete()

Deleted production: User.AgentSpace

Deleting class Agents.REST.Dispatch.AgentSpaceCleaned up production-owned artifacts for: AgentSpace


In [ ]:
Agent('Molly').delete()
try:
    molly('Hello')
except KeyError as e:
    print(e)


Deleting class Agents.Gateway.MollyService
Deleting class Agents.Process.Molly"No Agent found for 'Molly'"
